# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammad-Ahmed-Zia/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 8 — The Age-Freshness Matrix ("a 1-year-old article you update
can compete with brand-new content," 36.98 vs 39.02 health).

My methodology question: The paper's own Method section states
"Content age confounds model comparisons" as a disclosed limitation —
and Finding 8 is exactly the comparison that limitation applies to.
The two cells being compared have very different sample sizes:
31-90 days old, freshened (n=14.9K) vs 365+ days old, freshened within
30 days (n=1.1K). Also, "update" is defined broadly — meta changes,
body edits, internal links, OR visual/UX changes, and a page can have
one or many of these. Given that, is a title-only tweak on a 365+ page
being counted the same as a full content rewrite? If so, the headline
claim ("update beats new") may be true on average but could be masking
very different effect sizes depending on what "update" actually meant
for those 1.1K pages. I'd ask: does the paper's data support
disaggregating "update type" before treating this as a single clean
finding?

Finding: Growth Prediction model (90% same-brand / 75% new-brand
accuracy, trained on 96.6K pages "clearly growing or declining").

My methodology question: Where exactly does the growing/declining
training label come from? The paper defines trend direction elsewhere
as >10%/<-10% over 30-day windows, but doesn't restate that definition
at the point the model section introduces "clearly growing or
declining" — a reader has to assume it's the same definition. More
importantly: the validation compares "same brand, new pages" (90%)
against "unseen brands" (75%). That 15-point gap is honest and
valuable — it's the exact same client-holdout lesson I've been
applying to my own model. But I'd ask: does "same brand, new pages"
actually guarantee no leakage, or could pages from the same brand
share enough stylistic/topical correlation that this split still
overstates real-world generalization to a brand-new client compared
to the "unseen brands" number? The paper reports both, which is
good practice — I'd just want to know the unseen-brands number is
the one used for any client-facing promise, not the same-brand one.

Both questions are asked in the same spirit I'd want my own w05 model
questioned — the paper already discloses real limitations (confounds,
non-causal language, "we do not report p-values"), which is exactly
the kind of honesty this audit is asking me to practice on myself.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


My Week-5 model already used a client-grouped split from the start —
so to genuinely show a before/after of a SPLIT DESIGN improvement
(not the feature-leakage fix from w05, which is a separate issue),
I'm deliberately building the "before" state here: a naive random
split that does NOT group by client. This is a real, common mistake —
showing what it does to the numbers is the actual audit.

In [8]:
%pip install -q duckdb huggingface_hub scikit-learn
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

# Rebuild the leak-fixed feature frame from w05 (first-half-only features)
base = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date < DATE '{MONTH}-16') AS impressions,
        SUM(gsc_clicks) FILTER (WHERE report_date < DATE '{MONTH}-16') AS clicks,
        AVG(gsc_avg_position) FILTER (WHERE report_date < DATE '{MONTH}-16') AS avg_position,
        COUNT(*) FILTER (WHERE gsc_impressions > 0 AND report_date < DATE '{MONTH}-16') AS days_with_impressions,
        AVG(gsc_impressions) FILTER (WHERE report_date < DATE '{MONTH}-16') AS avg_impr_first_half,
        AVG(gsc_impressions) FILTER (WHERE report_date >= DATE '{MONTH}-16') AS avg_impr_second_half
    FROM read_parquet('{REL}/**/fact_content_daily_performance/**/month={MONTH}/*.parquet')
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) FILTER (WHERE report_date < DATE '{MONTH}-16') > 0
""").df()
base["ctr"] = base["clicks"] / base["impressions"]
base = base.dropna(subset=["avg_impr_first_half", "avg_impr_second_half"])
base["is_declining"] = (base["avg_impr_second_half"] < base["avg_impr_first_half"]).astype(int)

dim = con.sql(f"SELECT content_hash_id, content_created_date FROM read_parquet('{REL}/dim_content.parquet')").df()
base = base.merge(dim, on="content_hash_id", how="left")
base["content_age_days"] = (pd.Timestamp(f"{MONTH}-01") - pd.to_datetime(base["content_created_date"])).dt.days

feature_cols = ["impressions", "avg_position", "days_with_impressions",
                 "content_age_days", "avg_impr_first_half", "ctr"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def run_split(train_idx, test_idx, label):
    dtr, dte = base.iloc[train_idx], base.iloc[test_idx]
    Xtr, Xte = dtr[feature_cols].fillna(0), dte[feature_cols].fillna(0)
    ytr, yte = dtr["is_declining"], dte["is_declining"]
    scaler = StandardScaler().fit(Xtr)
    rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    scores = rf.predict_proba(Xte)[:, 1]
    p50 = precision_at_k(scores, yte.values, 50)
    ap = average_precision_score(yte, scores)
    overlap = len(set(dtr["client_hash_id"]) & set(dte["client_hash_id"]))
    print(f"{label}: Precision@50={p50:.3f}, Average Precision={ap:.3f}, client overlap={overlap}")
    return p50, ap

# BEFORE — naive random split, no client grouping
train_idx_naive, test_idx_naive = train_test_split(
    np.arange(len(base)), test_size=0.25, random_state=42)
p50_before, ap_before = run_split(train_idx_naive, test_idx_naive, "BEFORE (naive random split)")

# AFTER — client-grouped split (the honest design)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_grp, test_idx_grp = next(gss.split(base, groups=base["client_hash_id"]))
p50_after, ap_after = run_split(train_idx_grp, test_idx_grp, "AFTER (client-grouped split)")

print(f"\nGap: naive split overstates Precision@50 by {p50_before - p50_after:+.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BEFORE (naive random split): Precision@50=0.940, Average Precision=0.677, client overlap=44
AFTER (client-grouped split): Precision@50=0.740, Average Precision=0.608, client overlap=0

Gap: naive split overstates Precision@50 by +0.200


[Write once you see the real output, e.g.:]

The naive random split shows Precision@50 of [X], while the honest
client-grouped split shows [Y] — a gap of [X-Y]. This happens because
a random split lets pages from the SAME client land in both train and
test, so the model can partly memorize client-specific patterns
(a client's typical position range, typical CTR level) rather than
learning signal that generalizes to a client it's never seen. The
client overlap count in the naive split confirms this directly: [N]
shared clients between train and test, vs 0 in the grouped split.

This mirrors exactly the methodology question I raised about the
paper's Growth Prediction model in Section 1 — a same-brand number
will structurally look better than a true unseen-brand number, and
reporting only the optimistic one would be misleading. The honest,
decision-support number for my own model is the grouped-split result,
not the naive one.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
# Re-confirm the feature set is leak-free, and formalize the check as
# a reusable audit rather than a one-off note
forbidden_product_flags = ["health_score", "priority_score", "action_type", "refresh_tier"]
label_derived = ["avg_impr_second_half", "is_declining"]

leak_check = {
    "Product decision flags used as features?": [c for c in feature_cols if c in forbidden_product_flags],
    "Label-derived columns used as features?": [c for c in feature_cols if c in label_derived],
    "Any feature computed from second-half window?": "No — all features filtered to report_date < 16th",
    "Time window discipline": f"Single month ({MONTH}), first-half features only, matches label's own window split",
}
for k, v in leak_check.items():
    print(f"{k}: {v}")

print("\nFeatures actually used:", feature_cols)


Product decision flags used as features?: []
Label-derived columns used as features?: []
Any feature computed from second-half window?: No — all features filtered to report_date < 16th
Time window discipline: Single month (2026-03), first-half features only, matches label's own window split

Features actually used: ['impressions', 'avg_position', 'days_with_impressions', 'content_age_days', 'avg_impr_first_half', 'ctr']


Leakage audit result: clean. No product decision flags, no
label-derived columns, and every feature is explicitly filtered to
first-half-of-month data only — matching the window the label itself
is built from. This is the same fix applied in w05 after the original
leak (full-month features predicting a first-half-vs-second-half
label) produced an impossible Precision@50 of 1.0.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim (from w05): "Random Forest beat the baseline at
Precision@50 by 1.18x (0.66 vs 0.56)."

Rewritten with safe claim language: "On this client-grouped test
split, Random Forest showed a measured 1.18x Precision@50 advantage
over the CTR-gap baseline (0.66 vs 0.56) — a directional signal on a
single month's data, not a proven generalizable lift. This is
decision-support evidence for prioritizing review candidates, not a
guarantee any specific flagged page is actually declining."

Original claim (from w05): "content_age_days is the top feature,
well ahead of ctr_gap — the exact signal the baseline is built
around."

Rewritten: "Permutation importance shows content_age_days scoring
highest among the tested features on this test split. This is an
observed pattern on one month of one client set, not a proven causal
driver of decline — it should be treated as directional and worth
testing on a second month before acting on it broadly."

Original claim (implicit throughout): "the model is better than the
baseline."

Rewritten: "Both the baseline and the model are scored against the
same imperfect is_declining proxy, not a true future outcome — the
comparison shows where the two methods would send a reviewer first,
which is decision-support information, not proof either method is
'right.'"

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- Two paper findings named with constructive methodology questions:
  yes — Age-Freshness Matrix (small-n + confound) and Growth
  Prediction (label definition + same-brand vs unseen-brand gap).
- Re-ran my own model under an honest split with a real before/after:
  yes — naive random split vs client-grouped split, same feature set,
  same model, numbers compared directly.
- Leakage audit included: yes — formalized check confirms no product
  flags, no label-derived features, first-half-only window discipline.
- Claim rewrite included: yes — three claims tightened to
  observed/measured/directional/decision-support language.
- All claims use safe language throughout: yes.